# 5.23 Latent Dirichlet Allocation & topic models

**Lesson tagline.** Documents mix topics, and topics mix words.

LDA combines Dirichlet priors, latent topic assignments, and discrete word likelihoods. Collapsed Gibbs sampling alternates local topic choices using document-topic and topic-word counts. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 523
rng = np.random.default_rng(SEED)

def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=float)
    return matrix / matrix.sum(axis=1, keepdims=True)

def lda_doc_topic_mean(counts, alpha):
    counts = np.asarray(counts, dtype=float)
    return (counts + alpha) / (counts.sum() + len(counts) * alpha)

def topic_word_mean(counts, eta):
    counts = np.asarray(counts, dtype=float)
    return normalize_rows(counts + eta)

def collapsed_conditional(doc_topic_counts, topic_word_counts, topic_totals, word, alpha, eta):
    left = doc_topic_counts + alpha
    right = (topic_word_counts[:, word] + eta) / (topic_totals + topic_word_counts.shape[1] * eta)
    probs = left * right
    return probs / probs.sum()

def build_lda_ladder():
    ladder = []
    ladder.append({"name": "D1 tiny 2-topic", "K": 2, "V": 5, "docs": [[0, 1, 0, 2], [3, 4, 3, 4]], "alpha": 0.5, "eta": 0.1})
    ladder.append({"name": "D2 clean 4-topic", "K": 4, "V": 8, "docs": [[0, 1, 0, 1], [2, 3, 2, 3], [4, 5, 4, 5], [6, 7, 6, 7]], "alpha": 0.3, "eta": 0.1})
    ladder.append({"name": "D3 overlapping vocabulary", "K": 3, "V": 7, "docs": [[0, 1, 2, 2, 3], [2, 3, 4, 4], [0, 4, 5, 6]], "alpha": 0.5, "eta": 0.2})
    ladder.append({"name": "D4 inline real snippets", "K": 3, "V": 9, "docs": [[0, 1, 2, 0], [3, 4, 5, 4], [6, 7, 8, 6], [0, 4, 8]], "alpha": 0.4, "eta": 0.1})
    ladder.append({"name": "D5 sparse label-switching", "K": 5, "V": 12, "docs": [[0, 1, 0, 2, 10], [3, 4, 3, 5], [6, 7, 6, 8], [9, 10, 11, 9], [1, 5, 8, 11]], "alpha": 1.5, "eta": 0.05})
    return ladder

def initialize_assignments(docs, K, random_state):
    assignments = []
    for doc in docs:
        assignments.append(random_state.integers(0, K, size=len(doc)))
    return assignments

def count_assignments(docs, assignments, K, V):
    doc_topic = np.zeros((len(docs), K), dtype=int)
    topic_word = np.zeros((K, V), dtype=int)
    topic_total = np.zeros(K, dtype=int)
    for d, doc in enumerate(docs):
        for i, word in enumerate(doc):
            topic = assignments[d][i]
            doc_topic[d, topic] += 1
            topic_word[topic, word] += 1
            topic_total[topic] += 1
    return doc_topic, topic_word, topic_total

def run_lda_gibbs(rung, n_iter, random_state):
    docs = rung["docs"]
    K = rung["K"]
    V = rung["V"]
    alpha = rung["alpha"]
    eta = rung["eta"]
    assignments = initialize_assignments(docs, K, random_state)
    errors = []
    perplexities = []
    for _ in range(n_iter):
        doc_topic, topic_word, topic_total = count_assignments(docs, assignments, K, V)
        for d, doc in enumerate(docs):
            for i, word in enumerate(doc):
                old = assignments[d][i]
                doc_topic[d, old] -= 1
                topic_word[old, word] -= 1
                topic_total[old] -= 1
                probs = collapsed_conditional(doc_topic[d], topic_word, topic_total, word, alpha, eta)
                new = random_state.choice(K, p=probs)
                assignments[d][i] = new
                doc_topic[d, new] += 1
                topic_word[new, word] += 1
                topic_total[new] += 1
        theta = normalize_rows(doc_topic + alpha)
        phi = topic_word_mean(topic_word, eta)
        log_prob = 0.0
        count = 0
        for d, doc in enumerate(docs):
            for word in doc:
                pred = float(theta[d] @ phi[:, word])
                log_prob += np.log(pred + 1e-300)
                count += 1
        perplexities.append(float(np.exp(-log_prob / count)))
        errors.append(float(np.mean(np.abs(theta - 1.0 / K))))
    return assignments, theta, phi, errors, perplexities

## The concept, built once (D1)
LDA's joint model is
$$p(w,z,\theta,\phi)=\prod_k p(\phi_k\mid\eta)\prod_d p(\theta_d\mid\alpha)\prod_n p(z_{dn}\mid\theta_d)p(w_{dn}\mid\phi_{z_{dn}}).$$
Collapsed Gibbs uses document-topic counts times topic-word predictive probability for each local topic update.

In [ ]:
doc_counts = np.array([8, 2])
alpha = 0.5
doc_mean = lda_doc_topic_mean(doc_counts, alpha)
assert np.allclose(doc_mean, [8.5 / 11.0, 2.5 / 11.0])
assert np.allclose(doc_mean, [0.7727272727, 0.2272727273])
print("document-topic mean", doc_mean)

Topic-word rows normalize counts plus $\eta=0.1$. The Gibbs conditional for a word is proportional to a document-topic count factor times that topic-word probability, and the predictive word distribution $\theta_d^\top\phi$ must sum to $1$.

In [ ]:
eta = 0.1
topic_word_counts = np.array([[4, 1, 0], [0, 2, 3]])
phi = topic_word_mean(topic_word_counts, eta)
topic_totals = topic_word_counts.sum(axis=1)
cond = collapsed_conditional(np.array([8, 2]), topic_word_counts, topic_totals, 1, alpha, eta)
predictive = doc_mean @ phi
assert np.allclose(phi.sum(axis=1), [1.0, 1.0])
assert abs(predictive.sum() - 1.0) < 1e-12
assert np.all(cond > 0)
print("topic-word rows", phi)
print("Gibbs conditional", cond)
print("predictive sums to", predictive.sum())

## The dataset ladder
The F10 ladder uses inline corpora only: a tiny exact 2-topic corpus, clean 4-topic corpus, overlapping vocabulary, real-ish snippets encoded as tokens, and a sparse high-dimensional corpus with label switching risk.

In [ ]:
ladder = build_lda_ladder()
for i, rung in enumerate(ladder, start=1):
    token_count = sum(len(doc) for doc in rung["docs"])
    print(i, rung["name"], "K=", rung["K"], "V=", rung["V"], "tokens=", token_count)

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
lda_results = {}
print("rung | final perplexity | marginal mixture error")
for rung in ladder:
    assignments, theta, phi, errors, perplexities = run_lda_gibbs(rung, 60, rng)
    lda_results[rung["name"]] = {"theta": theta, "phi": phi, "errors": errors, "perplexities": perplexities, "assignments": assignments}
    print(f"{rung['name']} | {perplexities[-1]:.3f} | {errors[-1]:.3f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    theta = lda_results[rung["name"]]["theta"]
    ax.imshow(theta, aspect="auto", vmin=0, vmax=1)
    ax.set_title(rung["name"], fontsize=8)
    ax.set_xlabel("topic")
    ax.set_ylabel("doc")
for ax, rung in zip(axes[1], ladder):
    ax.plot(lda_results[rung["name"]]["perplexities"])
    ax.set_title("perplexity", fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
On D5, setting $lpha$ too large makes every document diffuse, and label switching means topic IDs cannot be compared directly across runs. The fix is to use a smaller $lpha$ for sparse documents and align topics by their top words before comparison.

In [ ]:
d5 = dict(ladder[-1])
diffuse = dict(d5)
diffuse["alpha"] = 5.0
sparse = dict(d5)
sparse["alpha"] = 0.2
_, theta_bad, phi_bad, errors_bad, ppl_bad = run_lda_gibbs(diffuse, 40, rng)
_, theta_good, phi_good, errors_good, ppl_good = run_lda_gibbs(sparse, 40, rng)
top_bad = np.argsort(-phi_bad, axis=1)[:, :3]
top_good = np.argsort(-phi_good, axis=1)[:, :3]
print("large alpha mean max topic", round(float(theta_bad.max(axis=1).mean()), 3), "perplexity", round(ppl_bad[-1], 3))
print("smaller alpha mean max topic", round(float(theta_good.max(axis=1).mean()), 3), "perplexity", round(ppl_good[-1], 3))
print("top-word alignment keys bad", top_bad)
print("top-word alignment keys good", top_good)

## Evaluate it + Practice
- **Metric.** Marginal topic-mixture error and held-out-style predictive perplexity.
- **No-skill baseline.** Assign every word in a document to the document's majority topic.
- **Cheap sanity check.** Rows of $	heta$ and $\phi$ must sum to 1 after smoothing.
- **Ablation.** Increase $lpha$ and watch document mixtures become diffuse.
- **Failure signals.** Perplexity rises, topics share the same top words, or topic IDs swap across seeds.

### Practice prompts
1. Hold out one token per document and compute predictive probability.
2. Decrease $\eta$ and inspect sparse topic-word rows.
3. Implement topic alignment by maximum top-word overlap.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here